In [1]:
import torchvision
import os
import numpy as np
import torch
from torch.utils.data import Subset

import torchvision.transforms as transforms
import torchvision.models as models

from torch.utils.data import DataLoader, Subset, ConcatDataset
from sklearn.metrics import classification_report, accuracy_score
from PIL import Image
from tqdm import tqdm

import json

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
PROJECT_ROOT = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "ProfessionAI_AIengineering/9. Generative AI/"
    "Project_Generative_AI"
)

# Load train and test datasets

In [4]:
# define transforms

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])



In [5]:
# load same indices of 30% training set data taken in 01_captioning notebook

dataset_train = torchvision.datasets.OxfordIIITPet(root = os.path.join(PROJECT_ROOT, "data", "raw"),
                                             split = "trainval",
                                             transform=transform_train,
                                             download = True
                                             )


train_small_idx = np.load(
    os.path.join(PROJECT_ROOT, "data", "splits", "train_small_indices.npy")
)

dataset_train_small = Subset(dataset_train, train_small_idx)

In [6]:
len(dataset_train)

3680

In [7]:
len(dataset_train_small)

1104

In [8]:
dataset_train_small[20]

(tensor([[[-0.5253, -0.5596, -0.6281,  ..., -0.9877, -1.0048, -1.0390],
          [-0.6109, -0.6109, -0.6281,  ..., -0.9705, -0.9877, -1.0390],
          [-0.7479, -0.7137, -0.6965,  ..., -0.9705, -0.9877, -1.0048],
          ...,
          [-0.7137, -0.7137, -0.6281,  ..., -1.0390, -1.0390, -1.0390],
          [-0.5253, -0.6281, -0.6623,  ..., -1.0048, -1.0048, -1.0048],
          [-0.3541, -0.4054, -0.4568,  ..., -1.1075, -1.0562, -1.0219]],
 
         [[-0.4076, -0.4426, -0.5126,  ..., -0.8452, -0.8102, -0.8102],
          [-0.4951, -0.4951, -0.5126,  ..., -0.8452, -0.8277, -0.7927],
          [-0.6352, -0.6001, -0.5826,  ..., -0.8452, -0.8277, -0.7927],
          ...,
          [-0.8277, -0.8277, -0.7402,  ..., -1.1078, -1.1078, -1.1078],
          [-0.6352, -0.7402, -0.7577,  ..., -1.0728, -1.0728, -1.0728],
          [-0.4426, -0.4776, -0.5476,  ..., -1.1253, -1.0728, -1.0378]],
 
         [[-0.3230, -0.3753, -0.4798,  ..., -0.9156, -0.9853, -1.0201],
          [-0.4101, -0.4275,

In [9]:
dataset_train_small[1009][0]

tensor([[[-2.1179, -2.0837, -2.0665,  ..., -1.6042, -1.5870, -1.5870],
         [-2.1179, -2.1179, -2.1179,  ..., -1.5870, -1.5528, -1.5357],
         [-1.9809, -2.0665, -2.1008,  ..., -1.4843, -1.4843, -1.4843],
         ...,
         [-0.0629, -0.2684, -0.3883,  ...,  0.1939,  0.7248,  0.6221],
         [-0.1314, -0.1143, -0.4568,  ...,  0.3309,  0.6392,  0.4851],
         [-0.0801, -0.0801, -0.4226,  ...,  0.3652,  0.6563,  0.4166]],

        [[-1.8431, -1.8256, -1.8431,  ..., -0.7927, -0.6352, -0.5651],
         [-1.8957, -1.9307, -1.9307,  ..., -0.7752, -0.6176, -0.5476],
         [-1.7731, -1.8782, -1.9482,  ..., -0.7402, -0.6176, -0.5476],
         ...,
         [-0.0224, -0.2500, -0.3725,  ...,  0.4328,  1.0105,  0.9055],
         [-0.2675, -0.1975, -0.5126,  ...,  0.6078,  0.9230,  0.7654],
         [-0.2675, -0.2150, -0.5301,  ...,  0.6779,  0.9580,  0.6779]],

        [[-1.8044, -1.8044, -1.8044,  ..., -0.5844, -0.4275, -0.3230],
         [-1.8044, -1.8044, -1.8044,  ..., -0

In [10]:
# load test dataset
dataset_test = torchvision.datasets.OxfordIIITPet(root = os.path.join(PROJECT_ROOT, "data", "raw"),
                                             split = "test",
                                             transform=transform_test,
                                             download = True
                                             )

In [11]:
# attach transforms
dataset_train_small.dataset.transform = transform_train
dataset_test.transform = transform_test

In [12]:
len(dataset_test)

3669

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [14]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

/usr/local/lib/python3.12/dist-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)


In [15]:
# define and attach transforms

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset_train_small.dataset.transform = transform_train
dataset_test.transform = transform_test


# Build Synthetic Dataset

In [16]:
# load metadata

CHECKPOINT_FILE = os.path.join(
    PROJECT_ROOT,
    "data",
    "synthetic",
    "generation_metadata_final.json"
)

with open(CHECKPOINT_FILE, "r") as f:
    generation_metadata = json.load(f)

In [17]:
generation_metadata

{'0': [{'image_path': '/content/drive/MyDrive/Colab Notebooks/ProfessionAI_AIengineering/9. Generative AI/Project_Generative_AI/data/synthetic/images/0_0.png',
   'class_name': 'Pomeranian',
   'prompt': 'A high-resolution realistic photograph of a Pomeranian, the pomeranian breed of dog is occupying the bed where i am sitting.',
   'source': 'synthetic'},
  {'image_path': '/content/drive/MyDrive/Colab Notebooks/ProfessionAI_AIengineering/9. Generative AI/Project_Generative_AI/data/synthetic/images/0_1.png',
   'class_name': 'Pomeranian',
   'prompt': 'A high-resolution realistic photograph of a Pomeranian, my pomeranian dog is seated on the bed.',
   'source': 'synthetic'}],
 '1': [{'image_path': '/content/drive/MyDrive/Colab Notebooks/ProfessionAI_AIengineering/9. Generative AI/Project_Generative_AI/data/synthetic/images/1_0.png',
   'class_name': 'Havanese',
   'prompt': 'A high-resolution realistic photograph of a Havanese, a havanese dog is positioned on a tennis court.',
   'sour

In [18]:
len(generation_metadata)

1104

In [19]:
generation_metadata['0'][0]

{'image_path': '/content/drive/MyDrive/Colab Notebooks/ProfessionAI_AIengineering/9. Generative AI/Project_Generative_AI/data/synthetic/images/0_0.png',
 'class_name': 'Pomeranian',
 'prompt': 'A high-resolution realistic photograph of a Pomeranian, the pomeranian breed of dog is occupying the bed where i am sitting.',
 'source': 'synthetic'}

In [20]:
generation_metadata['0'][1]

{'image_path': '/content/drive/MyDrive/Colab Notebooks/ProfessionAI_AIengineering/9. Generative AI/Project_Generative_AI/data/synthetic/images/0_1.png',
 'class_name': 'Pomeranian',
 'prompt': 'A high-resolution realistic photograph of a Pomeranian, my pomeranian dog is seated on the bed.',
 'source': 'synthetic'}

In [21]:
# class mapping
class_to_idx = dataset_train.class_to_idx

In [22]:
# create Synthetic Dataset class
class SyntheticDataset(torch.utils.data.Dataset):
    def __init__(self, metadata, class_to_idx, transform=None):
        self.samples = []
        self.transform = transform
        self.class_to_idx = class_to_idx

        for idx in metadata:
            for item in metadata[idx]:
                path = item["image_path"]
                class_name = item["class_name"]

                if class_name in class_to_idx:
                    label = class_to_idx[class_name]
                    self.samples.append((path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label


In [23]:
# create synthetic dataset
synthetic_dataset = SyntheticDataset(
    generation_metadata,
    class_to_idx,
    transform=transform_train
)


# Create Baseline and Augmented Datasets

In [24]:
train_baseline = dataset_train_small

In [25]:
train_augmented = ConcatDataset([
    dataset_train_small,
    synthetic_dataset
])

In [26]:
print("Baseline size:", len(train_baseline))
print("Synthetic size:", len(synthetic_dataset))
print("Augmented size:", len(train_augmented))

Baseline size: 1104
Synthetic size: 2181
Augmented size: 3285


# DataLoaders

In [27]:
BATCH_SIZE = 64

train_loader_baseline = DataLoader(
    train_baseline,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4
)

train_loader_augmented = DataLoader(
    train_augmented,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4
)

test_loader = DataLoader(
    dataset_test,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4
)


# Define Model

In [28]:
num_classes = 37

def create_model():
    model = models.resnet18(pretrained=True)
    model.fc = torch.nn.Linear(model.fc.in_features, num_classes)
    return model.to(device)


# Training Function

In [29]:
def train_model(model, train_loader, epochs=5):

    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    model.train()

    for epoch in range(epochs):
        running_loss = 0

        for images, labels in tqdm(train_loader):
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}")

    return model


# Evaluation Function

In [33]:
def evaluate_model(model, loader):

    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    report_dict = classification_report(all_labels, all_preds, output_dict=True)

    return accuracy, report_dict


# Train Baseline

In [31]:
model_baseline = create_model()
model_baseline = train_model(model_baseline, train_loader_baseline, epochs=5)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 44.7M/44.7M [00:00<00:00, 76.5MB/s]
100%|██████████| 18/18 [00:56<00:00,  3.16s/it]


Epoch 1/5, Loss: 3.0214


100%|██████████| 18/18 [00:03<00:00,  5.10it/s]


Epoch 2/5, Loss: 1.5012


...

100%|██████████| 18/18 [00:03<00:00,  5.56it/s]


Epoch 4/5, Loss: 0.5089


100%|██████████| 18/18 [00:03<00:00,  5.59it/s]

Epoch 5/5, Loss: 0.3078


Baseline Accuracy: 0.8222949032433906
              precision    recall  f1-score   support

           0       0.71      0.77      0.74        98
           1       0.67      0.87      0.76       100
           2       0.82      0.27      0.41       100
           3       0.80      0.82      0.81       100
           4       0.72      0.81      0.76       100
           5       0.73      0.73      0.73       100
           6       0.69      0.89      0.78       100
           7       0.90      0.93      0.92        88
           8       0.75      0.79      0.77        99
           9       0.84      0.72      0.77       100
          10       0.74      0.76      0.75       100
          11       0.85      0.87      0.86        97
          12       0.78      0.82      0.80       100
          13       0.86      0.79      0.82       100
          14       0.84      0.93      0.88       100
          15       0.98      0.88      0.93       100
          16       0.77      0.89      0.83

In [34]:
acc_baseline, report_baseline = evaluate_model(model_baseline, test_loader)

print("Baseline Accuracy:", acc_baseline)
print(report_baseline)


Baseline Accuracy: 0.8222949032433906
{'0': {'precision': 0.7142857142857143, 'recall': 0.7653061224489796, 'f1-score': 0.7389162561576355, 'support': 98.0}, '1': {'precision': 0.6744186046511628, 'recall': 0.87, 'f1-score': 0.759825327510917, 'support': 100.0}, '2': {'precision': 0.8181818181818182, 'recall': 0.27, 'f1-score': 0.40601503759398494, 'support': 100.0}, '3': {'precision': 0.7961165048543689, 'recall': 0.82, 'f1-score': 0.8078817733990148, 'support': 100.0}, '4': {'precision': 0.7232142857142857, 'recall': 0.81, 'f1-score': 0.7641509433962265, 'support': 100.0}, '5': {'precision': 0.73, 'recall': 0.73, 'f1-score': 0.73, 'support': 100.0}, '6': {'precision': 0.689922480620155, 'recall': 0.89, 'f1-score': 0.777292576419214, 'support': 100.0}, '7': {'precision': 0.9010989010989011, 'recall': 0.9318181818181818, 'f1-score': 0.9162011173184358, 'support': 88.0}, '8': {'precision': 0.75, 'recall': 0.7878787878787878, 'f1-score': 0.7684729064039408, 'support': 99.0}, '9': {'preci

# Train Augmented

In [32]:
model_augmented = create_model()
model_augmented = train_model(model_augmented, train_loader_augmented, epochs=5)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 52/52 [01:45<00:00,  2.03s/it]


Epoch 1/5, Loss: 1.8847


100%|██████████| 52/52 [00:13<00:00,  3.86it/s]


Epoch 2/5, Loss: 0.4982


100%|██████████| 52/52 [00:13<00:00,  3.86it/s]


...

100%|██████████| 52/52 [00:13<00:00,  3.93it/s]


Epoch 4/5, Loss: 0.1265


100%|██████████| 52/52 [00:13<00:00,  3.90it/s]

Epoch 5/5, Loss: 0.0682


Augmented Accuracy: 0.8560915780866721
              precision    recall  f1-score   support

           0       0.81      0.81      0.81        98
           1       0.75      0.78      0.76       100
           2       0.63      0.52      0.57       100
           3       0.96      0.79      0.87       100
           4       0.72      0.94      0.81       100
           5       0.74      0.91      0.82       100
           6       0.70      0.86      0.77       100
           7       0.73      0.97      0.83        88
           8       0.79      0.87      0.83        99
           9       0.94      0.79      0.86       100
          10       0.81      0.89      0.85       100
          11       0.96      0.89      0.92        97
          12       0.89      0.91      0.90       100
          13       0.94      0.89      0.91       100
          14       0.90      0.95      0.93       100
          15       0.95      0.95      0.95       100
          16       0.91      0.90      0.9

In [35]:
acc_augmented, report_augmented = evaluate_model(model_augmented, test_loader)

print("Augmented Accuracy:", acc_augmented)
print(report_augmented)

Augmented Accuracy: 0.8560915780866721
{'0': {'precision': 0.8061224489795918, 'recall': 0.8061224489795918, 'f1-score': 0.8061224489795918, 'support': 98.0}, '1': {'precision': 0.75, 'recall': 0.78, 'f1-score': 0.7647058823529411, 'support': 100.0}, '2': {'precision': 0.6341463414634146, 'recall': 0.52, 'f1-score': 0.5714285714285714, 'support': 100.0}, '3': {'precision': 0.9634146341463414, 'recall': 0.79, 'f1-score': 0.8681318681318682, 'support': 100.0}, '4': {'precision': 0.7175572519083969, 'recall': 0.94, 'f1-score': 0.8138528138528138, 'support': 100.0}, '5': {'precision': 0.7398373983739838, 'recall': 0.91, 'f1-score': 0.8161434977578476, 'support': 100.0}, '6': {'precision': 0.6991869918699187, 'recall': 0.86, 'f1-score': 0.7713004484304933, 'support': 100.0}, '7': {'precision': 0.7327586206896551, 'recall': 0.9659090909090909, 'f1-score': 0.8333333333333334, 'support': 88.0}, '8': {'precision': 0.7889908256880734, 'recall': 0.8686868686868687, 'f1-score': 0.8269230769230769,

# Final Result

In [43]:
# save metrics + experiment info
results = {
    "baseline": {
        "accuracy": acc_baseline,
        "report": report_baseline
    },
    "augmented": {
        "accuracy": acc_augmented,
        "report": report_augmented
    },
    "experiment_info": {
        "train_size_real": len(train_baseline),
        "train_size_synthetic": len(synthetic_dataset),
        "test_size": len(dataset_test),
        "epochs": 5,
        "batch_size": BATCH_SIZE,
        "model": "ResNet18"
    }
}


In [44]:
MODEL_DIR = os.path.join(PROJECT_ROOT, "models")
os.makedirs(MODEL_DIR, exist_ok=True)


with open(os.path.join(MODEL_DIR, "evaluation_results.json"), "w") as f:
    json.dump(results, f, indent=4)

In [45]:
# save models weights
torch.save(
    model_baseline.state_dict(),
    os.path.join(MODEL_DIR, "resnet18_baseline.pth")
)

torch.save(
    model_augmented.state_dict(),
    os.path.join(MODEL_DIR, "resnet18_augmented.pth")
)

In [39]:
with open(os.path.join(MODEL_DIR, "evaluation_results.json"), "r") as f:
    results = json.load(f)

print(results["baseline"]["accuracy"])
print(results["augmented"]["accuracy"])

0.8222949032433906
0.8560915780866721


# Conclusion

We observed a meaningful improvement in the 37-class fine-grained breed classification task through the use of Synthetic Data Augmentation.

The baseline model was trained on 1,104 real images (30% of the available training data), while the augmented model was trained on the same 1,104 real images plus 2,181 diffusion-generated synthetic images (approximately a 2× data augmentation factor).


The augmented model demonstrated:

- Improved generalization to unseen real test data

- Higher macro-averaged F1-score, indicating more balanced performance across breeds

- Higher weighted F1-score

- An increase in overall accuracy from 82.2% to 85.6%


Specifically, the augmented model achieved an accuracy improvement of 3.4% over the baseline trained on limited real data. Additionally, macro F1-score increased by approx. 3.7%, indicating improved performance across breeds.

These results indicate that diffusion-generated synthetic images effectively enriched the training distribution and enhanced the model’s ability to generalize to unseen samples. By increasing intra-class variability and introducing additional pose, texture, and appearance diversity, the synthetic data helped the model learn more robust and discriminative feature representations.